# Answer-flip study — Step 1b: port and gate FlashVID, HoliTom, LLaVA-Scissor (Run-All)

**Purpose.** Same idea as Step 1: check that each newly ported compressor behaves like its published version on LLaVA-OV-7B, using only the **pilot** Video-MME chunks (01, 05, 10, 15, 19). Also adds two **held-out noise passes** of the full model.

| Method | Settings | Code followed | Notes |
|---|---|---|---|
| FlashVID (ICLR 2026) | 5%, 10%, 25% | official repo, functions copied verbatim | 1.25× tokens before the LLM, pruned to 30% at layer 20 (average = the stated ratio) |
| HoliTom (NeurIPS 2025) | 5%, 10%, 25% | official repo, functions copied verbatim | outer merging + inner merging at layer 18 (r = 0.5) |
| LLaVA-Scissor (2025) | 5%, 10%, 25% | official repo, functions copied verbatim | its paper sets only a threshold τ; we search τ per video to hit the same budget |

All three were checked offline against the official code: identical outputs on test data (FlashVID and HoliTom compression, FlashVID's inner pruning, HoliTom's inner merging, Scissor's component search).

**Noise passes (new):** frames sampled a quarter interval later; answer options shown in reverse order (mapped back before comparing). Together with Step 1's two passes this gives four.

**Gate (written to `gate_rules.json` before any data):** same rule as Step 1 — within 2 points of a published relative accuracy for LLaVA-OV-7B on Video-MME, or the 95% video-clustered CI covers it.

## How to run
1. Settings → Accelerator **GPU T4 x2**, Internet **On**, HF_TOKEN in Add-ons → Secrets (already set).
2. *(Recommended)* **Add Input → Upload** the Step 1 `results.jsonl` as a small dataset. The analysis then joins the four noise passes per question and checks that the full model gives the same answers as in Step 1. Without it, everything still runs.
3. `QUICK_TEST = True` → **Run All** (~20 min). Check: `Compressor self-test OK ✅`, `Attention preflight OK ✅`, equivalence check `PASS` (now also checks our own decoder loop), a few ✓ lines.
4. `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**. Expect ~5–7 h for ~600 questions.
5. Send back `summary.json`, `gate_table.csv`, `step1b.png`, `gate_rules.json`, `equivalence_check.json`, `results.jsonl` (+ `errors.log` if not empty).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — only QUICK_TEST should be changed                       ║
# ╚══════════════════════════════════════════════════════════════════╝
QUICK_TEST = True   # True = ~20-min smoke test with a tiny model. False = the real Step 1b run.

MODEL_ID_REAL  = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
MODEL_ID_QUICK = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"

NUM_FRAMES      = 32
VIDEOMME_CHUNKS = ["01", "05", "10", "15", "19"]     # same PILOT chunks as Step 1 (never used in the confirmatory study)
MAX_Q_PER_CHUNK = None

METHOD_GRID = {
    "flashvid": [0.05, 0.10, 0.25],
    "holitom":  [0.05, 0.10, 0.25],
    "scissor":  [0.05, 0.10, 0.25],
}
# FlashVID official LLaVA-OV settings (scripts/llava_ov.sh): fixed for every ratio
FLASHVID = dict(alpha=0.7, token_selection_method="attn_div_v2", temporal_threshold=0.8, do_segment=True,
                segment_threshold=0.9, min_segment_num=8, complementary_segment=True,
                expansion=1.25, pruning_layer=20, llm_retention_ratio=0.3)
# HoliTom official LLaVA-OV settings (scripts/eval_ov-7b_holitom.sh); 5% is not in the script -> same T as 10%
HOLITOM = dict(T={0.05: 0.65, 0.10: 0.65, 0.25: 0.80}, beta=0.6, D=0.0, K=7, max_window=1024,
               inner_layer=18, inner_r=0.5)
SCISSOR = dict(epsilon=0.05, search_steps=12, refine_steps=6)   # tau searched per video to hit the token budget

# Held-out noise passes (new in Step 1b; Step 1 had a half-interval shift and a cyclic option shift)
NOISE_FRAME_SHIFT_Q = 0.25      # frames sampled a quarter interval later
NOISE_OPTION_ORDER = "reverse"  # options shown in reverse order (mapped back before comparing)

# Published Video-MME scores on LLaVA-OV-7B (32 frames), as relative accuracy in % of each paper's vanilla.
GATE_REFS = {
    ("flashvid", "0.1"):  {"values": [98.8],  "src": "FlashVID (arXiv 2602.08024) T1: 57.8 / 58.5"},
    ("flashvid", "0.25"): {"values": [101.2], "src": "FlashVID T1: 59.2 / 58.5"},
    ("holitom", "0.05"):  {"values": [92.5],  "src": "Du et al. CVPR 2026 (arXiv 2603.21957) T1: 54.2 / 58.6"},
    ("holitom", "0.1"):   {"values": [96.9],  "src": "HoliTom (arXiv 2505.21334) T2: 56.8 / 58.6 (Du et al. same)"},
    ("holitom", "0.25"):  {"values": [100.5], "src": "HoliTom T2: 58.9 / 58.6"},
    ("scissor", "0.05"):  {"values": [91.0],  "src": "Du et al. CVPR 2026 T1: 53.3 / 58.6"},
    ("scissor", "0.1"):   {"values": [95.2],  "src": "Du et al. CVPR 2026 T1: 55.8 / 58.6"},
}
GATE_TOL_POINTS = 2.0

SEED = 0
MAX_RUNTIME_HOURS = 10.5
OUT_DIR  = "/kaggle/working/flips_step1b"
WORK_DIR = "/tmp/astra_data"

if QUICK_TEST:
    MODEL_ID = MODEL_ID_QUICK
    NUM_FRAMES = 16
    VIDEOMME_CHUNKS = ["01"]
    MAX_Q_PER_CHUNK = 12
    OUT_DIR = OUT_DIR + "_quick"
else:
    MODEL_ID = MODEL_ID_REAL
print("Mode:", "QUICK TEST" if QUICK_TEST else "REAL STEP 1b RUN", "| model:", MODEL_ID, "| frames:", NUM_FRAMES)

### 1 · Setup

In [ ]:
# ── Setup: folders, environment, small installs ──────────────────────
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_home")      # keep HF cache on the big temp disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "WARN ") + "pip install " + " ".join(pkgs))

pip_install("hf_xet")   # faster Hugging Face downloads (harmless if already present)
pip_install("av")       # backup video decoder

try:
    from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401
    print("transformers OK")
except ImportError:
    pip_install("-U", "transformers", "accelerate")
    raise SystemExit("transformers was upgraded -> menu 'Run > Restart & Clear Cell Outputs', then 'Run All' again.")

### 1b · Gate rule and published numbers (written before any data)

In [ ]:
# ── Record the gate rule and reference numbers before any data is touched ──
import json, datetime, os
os.makedirs(OUT_DIR, exist_ok=True)
GATE_RECORD = {
    "written_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model": MODEL_ID, "frames": NUM_FRAMES, "chunks": VIDEOMME_CHUNKS,
    "rule": (f"A (method, setting) passes if its relative accuracy (method acc / full-token acc on the same questions) "
             f"is within {GATE_TOL_POINTS} points of a published value, or its 95% video-clustered bootstrap CI "
             f"covers a published value. Failing methods are excluded from the main study and reported."),
    "references_percent": {f"{m}@{s}": v for (m, s), v in GATE_REFS.items()},
    "methods": METHOD_GRID, "settings": {"flashvid": FLASHVID, "holitom": {**HOLITOM, "T": {str(k): v for k, v in HOLITOM["T"].items()}}, "scissor": SCISSOR},
    "noise_floor": {"frame_shift_q": NOISE_FRAME_SHIFT_Q, "option_order": NOISE_OPTION_ORDER},
}
path = os.path.join(OUT_DIR, "gate_rules.json")
if not os.path.exists(path):
    with open(path, "w") as f:
        json.dump(GATE_RECORD, f, indent=2)
print(json.dumps(GATE_RECORD, indent=2))

### 2 · Imports and GPU check

In [ ]:
# ── Imports + hardware check ──────────────────────────────────────────
import json, math, time, random, shutil, zipfile, gc, heapq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
import transformers
from transformers import LlavaOnevisionForConditionalGeneration, AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "No GPU found! Right panel > Settings > Accelerator > 'GPU T4 x2', then Run All again."
for i in range(N_GPU):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk for temp videos: {free_gb:.1f} GB")
if free_gb < 12:
    print("⚠️ Less than 12 GB free — a 5 GB video chunk needs ~11 GB while unzipping.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Optional: a Hugging Face token stored as a Kaggle Secret named HF_TOKEN (Add-ons > Secrets).
# Not needed for these public repos, but harmless.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret (fine for public repos)")

### 3 · Video-MME loader (+ frame shift)

In [ ]:
# ── Data: Video-MME questions + video chunks + frame reading ────────────
VMME_REPOS = ["lmms-eval/Video-MME", "lmms-lab/Video-MME"]

def _pick_repo():
    for repo in VMME_REPOS:
        try:
            return repo, list_repo_files(repo, repo_type="dataset")
        except Exception as e:
            print(f"Repo {repo} not reachable: {type(e).__name__}: {e}")
    raise RuntimeError("Cannot reach Video-MME on Hugging Face. Is Internet ON? (Settings > Internet)")

VMME_REPO, VMME_FILES = _pick_repo()
print("Using", VMME_REPO, "|", len(VMME_FILES), "files")

def load_videomme_table():
    pq = [f for f in VMME_FILES if f.endswith(".parquet")]
    assert pq, "No question table (.parquet) found in the Video-MME repo"
    df = pd.concat([pd.read_parquet(hf_hub_download(VMME_REPO, f, repo_type="dataset", local_dir=WORK_DIR))
                    for f in pq], ignore_index=True)
    low = {c.lower(): c for c in df.columns}
    def col(*names):
        for n in names:
            if n in low:
                return low[n]
        return None
    c_key, c_vid = col("videoid", "video_id"), col("video_id", "videoid")
    c_q, c_opt, c_ans = col("question"), col("options"), col("answer")
    assert all([c_key, c_q, c_opt, c_ans]), f"Unexpected columns: {list(df.columns)}"
    c_qid, c_dur, c_task = col("question_id"), col("duration"), col("task_type")
    out = pd.DataFrame({
        "video_key": df[c_key].astype(str),
        "video_id":  df[c_vid].astype(str),
        "qid":       df[c_qid].astype(str) if c_qid else df.index.astype(str),
        "duration":  df[c_dur].astype(str) if c_dur else "unknown",
        "task_type": df[c_task].astype(str) if c_task else "unknown",
        "question":  df[c_q].astype(str),
        "options":   df[c_opt].apply(lambda o: [str(x) for x in list(o)]),
        "answer":    df[c_ans].astype(str).str.strip().str[0].str.upper(),
    })
    print(f"Video-MME table: {len(out)} questions, {out.video_key.nunique()} videos")
    return out

VIDEO_EXT = (".mp4", ".mkv", ".webm", ".avi", ".mov")

def fetch_chunk(chunk):
    """Download one ~5 GB zip, extract only the videos, delete the zip. Returns {file stem: path}."""
    fn = f"videos_chunked_{chunk}.zip"
    assert fn in VMME_FILES, f"{fn} not in repo"
    t0 = time.time()
    zpath = hf_hub_download(VMME_REPO, fn, repo_type="dataset", local_dir=WORK_DIR)
    vdir = os.path.join(WORK_DIR, f"videos_{chunk}")
    os.makedirs(vdir, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(VIDEO_EXT):
                with z.open(m) as src, open(os.path.join(vdir, os.path.basename(m.filename)), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove(zpath)
    shutil.rmtree(os.path.join(os.environ["HF_HOME"], "xet"), ignore_errors=True)
    vids = {os.path.splitext(f)[0]: os.path.join(vdir, f) for f in os.listdir(vdir)}
    print(f"Chunk {chunk}: {len(vids)} videos ready in {time.time()-t0:.0f}s "
          f"(free disk {shutil.disk_usage(WORK_DIR).free/1e9:.1f} GB)")
    return vids

def cleanup_chunk(chunk):
    shutil.rmtree(os.path.join(WORK_DIR, f"videos_{chunk}"), ignore_errors=True)

def _frame_indices(total, n, shift):
    """Uniform sampling as in LMMs-Eval (np.linspace over all frames). shift = fraction of one sampling
    interval to move every sample later (0.5 = the noise-floor 'half-interval frame shift')."""
    idx = np.linspace(0, total - 1, n)
    if shift:
        idx = idx + shift * (total - 1) / max(n - 1, 1)
    return np.clip(idx, 0, total - 1).round().astype(int)

def _read_cv2(path, n, shift=0.0):
    import cv2
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total > 0:
        for k in _frame_indices(total, n, shift):
            ok = False
            for off in (0, -1, 1, -5, 5, -20):
                kk = int(np.clip(k + off, 0, total - 1))
                cap.set(cv2.CAP_PROP_POS_FRAMES, kk)
                ok, fr = cap.read()
                if ok:
                    break
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def _read_av(path, n, shift=0.0):
    import av
    out = []
    with av.open(path) as c:
        s = c.streams.video[0]
        dur = float(s.duration * s.time_base) if s.duration else float(c.duration or 0) / 1e6
        end = max(dur - 0.5, 0)
        ts = np.linspace(0, end, n)
        if shift:
            ts = np.clip(ts + shift * end / max(n - 1, 1), 0, end)
        for t in ts:
            c.seek(int(t / s.time_base), stream=s, any_frame=False, backward=True)
            for fr in c.decode(s):
                out.append(fr.to_ndarray(format="rgb24"))
                break
    return out

def read_frames(path, n, shift=0.0):
    frames = []
    try:
        frames = _read_cv2(path, n, shift)
    except Exception as e:
        print("cv2 failed:", e)
    if len(frames) < n:
        try:
            frames = _read_av(path, n, shift)
        except Exception as e:
            print("av failed:", e)
    if len(frames) == 0:
        raise RuntimeError(f"Could not decode {path}")
    while len(frames) < n:          # pad rare short decodes
        frames.append(frames[-1])
    return frames[:n]

### 4 · Model + T4 memory fix + our decoder loop (inner-LLM pruning/merging)

In [ ]:
# ── Model: LLaVA-OneVision with direct control over visual tokens ──────
# ── T4 memory fix + safe attention routing ─────────────────────────────
# 1) Expand grouped K/V heads ourselves so PyTorch can use the memory-efficient kernel (math kernel -> OOM on T4).
# 2) Never hand an explicit mask to the memory-efficient kernel: on T4 that path can crash with
#    "illegal memory access". A plain causal mask is converted to is_causal=True (identical maths).
# 3) Guard: if any NON-causal mask (e.g. transformers mistaking position gaps for packed sequences)
#    reaches a long attention call, stop loudly instead of computing something silently wrong.
from torch.nn.attention import sdpa_kernel, SDPBackend
if not hasattr(torch.nn.functional, "_astra_orig_sdpa"):
    torch.nn.functional._astra_orig_sdpa = torch.nn.functional.scaled_dot_product_attention
_orig_sdpa = torch.nn.functional._astra_orig_sdpa
_TRIL_CACHE = {}
def _plain_causal(mask, L, S):
    if L != S or mask.dim() != 4 or mask.shape[0] != 1 or mask.shape[1] != 1:
        return False
    key = (L, str(mask.device))
    if key not in _TRIL_CACHE:
        _TRIL_CACHE.clear()
        _TRIL_CACHE[key] = torch.ones(L, L, dtype=torch.bool, device=mask.device).tril()
    allowed = mask[0, 0] if mask.dtype == torch.bool else (mask[0, 0] == 0)
    return torch.equal(allowed, _TRIL_CACHE[key])

def _sdpa_safe(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False, **kw):
    if enable_gqa and key.shape[-3] != query.shape[-3]:
        rep = query.shape[-3] // key.shape[-3]
        key = key.repeat_interleave(rep, dim=-3)
        value = value.repeat_interleave(rep, dim=-3)
    if attn_mask is not None:
        L, S = query.shape[-2], key.shape[-2]
        if _plain_causal(attn_mask, L, S):
            attn_mask, is_causal = None, True
        elif L > 1024:
            raise RuntimeError("A non-causal attention mask reached a long attention call — stopping instead of "
                               "computing silently wrong results. Send me this error.")
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

torch.nn.functional.scaled_dot_product_attention = _sdpa_safe
NO_MATH = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION]
print("T4 attention fix applied (K/V heads expanded, causal masks routed to is_causal, math kernel disabled for the LLM)")

max_memory = {i: "13GiB" for i in range(N_GPU)}
if N_GPU >= 2:
    max_memory[0] = "10GiB"
_dtype_kw = {"dtype": torch.float16} if int(transformers.__version__.split(".")[0]) >= 5 else {"torch_dtype": torch.float16}
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", max_memory=max_memory, attn_implementation="sdpa", **_dtype_kw).eval()
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def _get(root, paths):
    for p in paths:
        obj, ok = root, True
        for part in p.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError(f"None of {paths} found — send me the error and transformers version")

vision_tower = _get(model, ["vision_tower", "model.vision_tower"])
projector    = _get(model, ["multi_modal_projector", "model.multi_modal_projector"])
NEWLINE      = _get(model, ["image_newline", "model.image_newline"]).data
lm_head      = _get(model, ["lm_head", "language_model.lm_head"])
body = None
for _, mod in model.named_modules():
    if hasattr(mod, "embed_tokens") and hasattr(mod, "layers") and hasattr(mod, "norm"):
        body = mod
        break
assert body is not None, "Could not find the language decoder"

CFG, VCFG, TCFG = model.config, model.config.vision_config, model.config.text_config
VFL = CFG.vision_feature_layer
VSS = getattr(CFG, "vision_feature_select_strategy", "full")
IMG_SIZE, PATCH = VCFG.image_size, VCFG.patch_size
SIDE = IMG_SIZE // PATCH
POOLED = math.ceil(SIDE / 2)
NLAYERS, HID = TCFG.num_hidden_layers, TCFG.hidden_size
NHEADS, NKV = TCFG.num_attention_heads, TCFG.num_key_value_heads
HEAD_DIM = getattr(TCFG, "head_dim", None) or HID // NHEADS
INTER = TCFG.intermediate_size
EMB_DEVICE = body.embed_tokens.weight.device
LM_DEVICE = lm_head.weight.device
try:
    from transformers import AutoImageProcessor
    _ip = AutoImageProcessor.from_pretrained(MODEL_ID)
    IMG_MEAN, IMG_STD = list(_ip.image_mean), list(_ip.image_std)
except Exception:
    IMG_MEAN, IMG_STD = [0.5] * 3, [0.5] * 3

# ── FlashVID / HoliTom read the attention of the vision layer that produces the features ──
# (official code: attentions[-1].mean(heads).mean(queries), then the same 2x2 bilinear pooling as the features)
VDEV = next(vision_tower.parameters()).device
ENC_LAYERS = _get(vision_tower, ["vision_model.encoder.layers", "encoder.layers", "model.encoder.layers"])
assert isinstance(VFL, int), "Ports assume a single vision feature layer"
_n_hs = len(ENC_LAYERS) + 1
_h_idx = VFL if VFL >= 0 else _n_hs + VFL
VZ_LAYER, VZ_IN = ENC_LAYERS[_h_idx - 1], _h_idx - 1
VZ_HEADS = VCFG.num_attention_heads
VZ_HD = VCFG.hidden_size // VZ_HEADS
print(f"Vision attention read from vision layer {_h_idx - 1} of {len(ENC_LAYERS)}")
print(f"LLM layers={NLAYERS} hidden={HID} heads={NHEADS}/{NKV} | {SIDE}x{SIDE} -> {POOLED}x{POOLED} tokens/frame")

LETTERS = "ABCDEFGH"
LETTER_IDS = []
for L in LETTERS:
    ids = tok.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"Letter {L} is not a single token"
    LETTER_IDS.append(ids[0])

PREFIX = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"
PREFIX_IDS = tok(PREFIX, add_special_tokens=False, return_tensors="pt").input_ids[0]

def build_suffix_ids(question, options):
    opts = "\n".join(str(o) for o in options)
    s = ("\nSelect the best answer to the following multiple-choice question based on the video. "
         "Respond with only the letter (A, B, C, or D) of the correct option.\n"
         f"{question}\n{opts}\nThe best answer is:<|im_end|>\n<|im_start|>assistant\n")
    return tok(s, add_special_tokens=False, return_tensors="pt").input_ids[0]

def preprocess(frames):
    arr = np.stack([np.asarray(Image.fromarray(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))
                    for f in frames]).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(0, 3, 1, 2)
    m, s = torch.tensor(IMG_MEAN)[:, None, None], torch.tensor(IMG_STD)[:, None, None]
    return (x - m) / s

@torch.no_grad()
def encode_video(pix, bs=8):
    """frames -> X: (T, POOLED*POOLED, D) projected tokens (exactly LLaVA-OV's video path, before the newline token)
    and VA: (T, POOLED*POOLED) attention each patch receives in the vision layer that produces the features
    (mean over heads and queries, bilinear-pooled) — the signal FlashVID and HoliTom read from the vision tower."""
    pdev = next(projector.parameters()).device
    feats, vz_a = [], []
    for i in range(0, pix.shape[0], bs):
        out = vision_tower(pixel_values=pix[i:i+bs].to(VDEV, torch.float16), output_hidden_states=True)
        h = out.hidden_states[VFL]
        if VSS == "default":
            h = h[:, 1:]
        xin = out.hidden_states[VZ_IN].to(VZ_LAYER.layer_norm1.weight.device)
        if VSS == "default":
            xin = xin[:, 1:]
        xin = VZ_LAYER.layer_norm1(xin)
        b, P, _ = xin.shape
        q = VZ_LAYER.self_attn.q_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        k = VZ_LAYER.self_attn.k_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        w = torch.softmax(torch.matmul(q, k.transpose(2, 3)) * VZ_HD ** -0.5, dim=-1)
        vz_a.append(w.mean(dim=1).mean(dim=1))
        del q, k, w
        feats.append(projector(h.to(pdev)))
    def pool(t):                                        # (T, SIDE*SIDE, C) -> (T, POOLED*POOLED, C), = get_2dPool
        T, P, C = t.shape
        t = t.view(T, SIDE, SIDE, C).permute(0, 3, 1, 2).float()
        t = F.interpolate(t, size=(POOLED, POOLED), mode="bilinear")
        return t.permute(0, 2, 3, 1).reshape(T, POOLED * POOLED, C)
    f = pool(torch.cat(feats)).contiguous().to(torch.float16)
    va = pool(torch.cat(vz_a).unsqueeze(-1).to(f.device))[..., 0]
    return f, va

def llm_flops_layers(ns):
    """Prefill FLOPs of the LLM when layer l sees ns[l] tokens."""
    dkv = NKV * HEAD_DIM
    return float(sum(2 * (2 * n * HID * HID + 2 * n * HID * dkv + 2 * n * n * HID + 3 * n * HID * INTER) for n in ns))

def llm_flops(n):
    return llm_flops_layers([n] * NLAYERS)

def _sync():
    for i in range(N_GPU):
        torch.cuda.synchronize(i)

# ── inner-LLM operations (FlashVID: FastV-style pruning; HoliTom: similarity merging) ──
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb, repeat_kv

@torch.no_grad()
def last_query_attention(layer, h, pe):
    """Attention of the LAST token over all tokens in this layer, (1, heads, 1, N) — exactly the row both official
    codes read (FlashVID computes it this way under flash/sdpa; HoliTom takes the last row of eager attention)."""
    a = layer.self_attn
    dev = a.q_proj.weight.device
    h = h.to(dev)
    cos, sin = pe[0].to(dev), pe[1].to(dev)
    hn = layer.input_layernorm(h)
    q = a.q_proj(hn[:, -1:]).view(1, 1, -1, HEAD_DIM).transpose(1, 2)
    k = a.k_proj(hn).view(1, hn.shape[1], -1, HEAD_DIM).transpose(1, 2)
    q, _ = apply_rotary_pos_emb(q, q, cos[:, -1:], sin[:, -1:])
    _, k = apply_rotary_pos_emb(k, k, cos, sin)
    k = repeat_kv(k, NHEADS // NKV)
    w = torch.matmul(q, k.transpose(2, 3)) * a.scaling
    return torch.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)

@torch.no_grad()
def apply_inner(h, pos, pe, attn, inner, v0, vlen, has_newline):
    """Returns pruned/merged (h, pos, pe) and the number of visual tokens left."""
    dev = h.device
    a = attn.mean(dim=1)[0, -1].to(dev)                       # mean over heads -> (N,)
    N = h.shape[1]
    if inner["kind"] == "fastv":                               # FlashVID utils.fastv_prune
        n_keep = math.ceil(vlen * inner["keep_ratio"])
        top = torch.topk(a[v0:v0 + vlen], k=n_keep).indices.sort().values + v0
        keep = torch.cat([torch.arange(v0, device=dev), top, torch.arange(v0 + vlen, N, device=dev)]).sort().values
        n_vis = n_keep
    elif inner["kind"] == "holitom":                           # HoliTom modeling_qwen2 (newline counted in the span)
        L = vlen + (1 if (has_newline and inner["include_newline"]) else 0)
        n_top = round(L * (1 - inner["merge_ratio"]))
        top = a[v0:v0 + L].topk(n_top).indices + v0
        span = torch.arange(v0, v0 + L, device=dev)
        non_top = span[~torch.isin(span, top)]                 # ascending, as in the official loop
        hs = h[0].float()
        nt, tp = hs[non_top], hs[top]
        sim = (nt @ tp.t()) / (nt.norm(dim=-1, keepdim=True) * tp.norm(dim=-1, keepdim=True).t())
        tgt = sim.argmax(dim=-1)
        # official: for each non-top token in order, h[t] = (h[t] + h[i]) / 2  -> closed form below (same maths)
        m = torch.bincount(tgt, minlength=len(top))
        srt, perm = torch.sort(tgt, stable=True)
        first = torch.searchsorted(srt, srt)
        rank = torch.empty_like(tgt)
        rank[perm] = torch.arange(len(tgt), device=dev) - first + 1
        w = torch.pow(2.0, -(m[tgt] - rank + 1).float())
        new_top = tp * torch.pow(2.0, -m.float()).unsqueeze(1)
        new_top.index_add_(0, tgt, nt * w.unsqueeze(1))
        h = h.clone()
        h[0, top] = new_top.to(h.dtype)
        keep = torch.cat([torch.arange(v0, device=dev), top, torch.arange(v0 + L, N, device=dev)]).sort().values
        n_vis = n_top - (int(bool((top == v0 + vlen).any())) if L > vlen else 0)
    else:
        raise ValueError(inner["kind"])
    kp = keep.to(pos.device)
    pe = (pe[0][:, keep.to(pe[0].device)], pe[1][:, keep.to(pe[1].device)])
    return h[:, keep], pos[:, kp], pe, n_vis

@torch.no_grad()
def forward_layers(x, pos, inner=None, v0=0, vlen=0, has_newline=True):
    """Our own decoder loop (same calls as Qwen2Model.forward); lets FlashVID/HoliTom act inside the LLM."""
    pe = body.rotary_emb(x, pos)
    h, attn, ns, nvis = x, None, [], []
    vis_now = vlen
    for li, layer in enumerate(body.layers):
        if inner is not None and li == inner["layer"]:
            h, pos, pe, vis_now = apply_inner(h, pos, pe, attn, inner, v0, vlen, has_newline)
            attn = None
        if inner is not None and li == inner["layer"] - 1:
            attn = last_query_attention(layer, h, pe)
        ns.append(h.shape[1]); nvis.append(vis_now)
        h = layer(h, attention_mask=None, position_ids=pos, position_embeddings=pe, use_cache=False)
        if isinstance(h, tuple):
            h = h[0]
    h = body.norm(h.to(body.norm.weight.device))
    return h, ns, nvis

@torch.no_grad()
def llm_pass(vis, vis_pos, n_video_full, suf_ids, n_opt, pre_ids=None, newline=True, inner=None, manual=False):
    """One prefill. vis: kept video tokens (K, D); vis_pos: their position offsets inside the video span;
    n_video_full: length of the video span for positions (the newline and question follow it).
    newline=False drops LLaVA-OV's video newline token (LLaVA-Scissor's flat path).
    inner: optional FlashVID/HoliTom inner-LLM operation (runs our own decoder loop)."""
    E = body.embed_tokens
    pre_ids = PREFIX_IDS if pre_ids is None else pre_ids
    pre, suf = E(pre_ids.to(EMB_DEVICE)), E(suf_ids.to(EMB_DEVICE))
    parts = [pre, vis.to(EMB_DEVICE, pre.dtype)]
    if newline:
        parts.append(NEWLINE.to(EMB_DEVICE, pre.dtype)[None])
    parts.append(suf)
    x = torch.cat(parts, 0)[None]
    n_pre, n_q = pre.shape[0], suf.shape[0]
    vp = torch.as_tensor(vis_pos, device=EMB_DEVICE, dtype=torch.long)
    nl = 1 if newline else 0
    pos = torch.cat([torch.arange(n_pre, device=EMB_DEVICE), n_pre + vp] +
                    ([torch.tensor([n_pre + n_video_full], device=EMB_DEVICE)] if newline else []) +
                    [n_pre + n_video_full + nl + torch.arange(n_q, device=EMB_DEVICE)])[None]
    am = torch.ones(1, x.shape[1], dtype=torch.long, device=EMB_DEVICE)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        if inner is None and not manual:
            out = body(inputs_embeds=x, attention_mask=am, position_ids=pos, use_cache=False)
            h = out[0][0, -1]
            ns, nvis = [x.shape[1]] * NLAYERS, [vis.shape[0]] * NLAYERS
        else:
            hh, ns, nvis = forward_layers(x, pos, inner, v0=n_pre, vlen=vis.shape[0], has_newline=newline)
            h = hh[0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]),
            "ntok": int(vis.shape[0]), "vis_mean_layers": float(np.mean(nvis)), "sec": float(dt),
            "flops": llm_flops_layers(ns),
            "peak_gb": round(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9, 3)}

@torch.no_grad()
def preflight_attention(n_tok):
    dev = body.layers[0].self_attn.q_proj.weight.device
    q = torch.randn(1, NHEADS, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    kv = torch.randn(1, NKV, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    mask = torch.ones(n_tok, n_tok, dtype=torch.bool, device=dev).tril()[None, None]
    torch.cuda.reset_peak_memory_stats(dev)
    base = torch.cuda.memory_allocated(dev)
    try:
        with sdpa_kernel(NO_MATH):
            o1 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, is_causal=True, enable_gqa=True)
            o2 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, attn_mask=mask, enable_gqa=True)  # routed to is_causal
        torch.cuda.synchronize(dev)          # surface any asynchronous CUDA error HERE, not in a later cell
        assert torch.equal(o1, o2), "causal-mask routing changed the result"
    except RuntimeError as e:
        raise RuntimeError(f"Memory-efficient attention is not available here ({e}). Send me this message.")
    del o1, o2
    peak = (torch.cuda.max_memory_allocated(dev) - base) / 1e9
    del q, kv, mask
    torch.cuda.empty_cache()
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU{i}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    print(f"Attention preflight OK ✅  {n_tok} tokens -> extra peak memory {peak:.2f} GB")

preflight_attention(NUM_FRAMES * POOLED ** 2 + 400)
print("Model ready ✅")

### 4b · Safety check vs the official model forward (runs on the first video)

In [ ]:
# ── Safety check: our token pipeline must reproduce the official model forward ──
from transformers import AutoProcessor

def equivalence_check(frames, row):
    report = {"transformers": transformers.__version__}
    try:
        proc = AutoProcessor.from_pretrained(MODEL_ID)
        opts = "\n".join(str(o) for o in row.options)
        text = ("Select the best answer to the following multiple-choice question based on the video. "
                "Respond with only the letter (A, B, C, or D) of the correct option.\n"
                f"{row.question}\n{opts}\nThe best answer is:")
        conv = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": text}]}]
        prompt = proc.apply_chat_template(conv, add_generation_prompt=True)
        inputs = proc(text=prompt, videos=[np.stack(frames)], return_tensors="pt")
        inputs = {k: (v.to(EMB_DEVICE, torch.float16) if torch.is_floating_point(v) else v.to(EMB_DEVICE))
                  for k, v in inputs.items()}
        n_opt = min(len(row.options), len(LETTERS))
        with torch.no_grad(), sdpa_kernel(NO_MATH):
            try:
                out = model(**inputs, logits_to_keep=1)
            except TypeError:
                out = model(**inputs, num_logits_to_keep=1)
        lp_off = torch.log_softmax(out.logits[0, -1].float()[LETTER_IDS[:n_opt]], -1).cpu().numpy()
        del out
        ids = inputs["input_ids"][0].cpu()
        vid_tok = getattr(CFG, "video_token_id", None) or getattr(CFG, "video_token_index", None)
        posi = (ids == vid_tok).nonzero().flatten()
        X = encode_video(inputs["pixel_values_videos"][0])[0]
        vis = X.reshape(-1, X.shape[-1])
        report["n_video_tokens_official"] = int(len(posi))
        report["n_video_tokens_ours"] = int(vis.shape[0] + 1)
        ours = llm_pass(vis, np.arange(vis.shape[0]), vis.shape[0], ids[posi[-1] + 1:], n_opt, pre_ids=ids[: posi[0]])
        lp_ours = np.array(ours["logp"])
        report["logp_official"] = lp_off.round(4).tolist()
        report["logp_ours_same_pixels"] = lp_ours.round(4).tolist()
        report["max_abs_gap"] = float(np.abs(lp_off - lp_ours).max())
        # our own decoder loop (used by FlashVID / HoliTom) must give the same answer as the library forward
        man = llm_pass(vis, np.arange(vis.shape[0]), vis.shape[0], ids[posi[-1] + 1:], n_opt, pre_ids=ids[: posi[0]],
                       manual=True)
        report["logp_ours_manual_loop"] = man["logp"]
        report["max_abs_gap_manual_loop"] = float(np.abs(np.array(man["logp"]) - lp_ours).max())
        report["pixel_mean_abs_diff_ours_vs_official"] = float(
            (preprocess(frames) - inputs["pixel_values_videos"][0].float().cpu()).abs().mean())
        ok = (report["n_video_tokens_official"] == report["n_video_tokens_ours"] and report["max_abs_gap"] <= 0.15
              and report["max_abs_gap_manual_loop"] <= 0.05)
        report["status"] = "PASS" if ok else "FAIL"
    except Exception as e:
        report["status"] = "SKIPPED"
        report["error"] = f"{type(e).__name__}: {e}"
    with open(os.path.join(OUT_DIR, "equivalence_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Equivalence check:", json.dumps(report, indent=2))
    if report["status"] == "FAIL" or (report["status"] == "SKIPPED" and not QUICK_TEST):
        raise RuntimeError("Equivalence check failed/skipped — stopping so no GPU hours are wasted. "
                           "Send me equivalence_check.json and the log.")
    gc.collect(); torch.cuda.empty_cache()

### 5 · Compressors: FlashVID, HoliTom, LLaVA-Scissor (+ self-test)

In [ ]:
# ══ Compressors (Step 1b): FlashVID, HoliTom, LLaVA-Scissor ══
# Each compressor returns a dict: tokens (K, D) entering the LLM, pos (their position ids, contiguous for all three
# as in the official code), newline (whether LLaVA-OV's video newline token is appended), and inner (an optional
# inner-LLM operation applied by llm_pass at a given layer).

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# FlashVID (Fan et al., ICLR 2026 oral) — github.com/Fanziyang-v/FlashVID, flashvid/utils.py + token_selection.py
# MIT License, Copyright (c) the FlashVID authors. The functions below are copied verbatim (only the config class is
# re-declared here); the official LLaVA-OV path passes the bilinear-pooled features and the bilinear-pooled
# last-vision-layer attention (mean over heads and queries), then prunes inside the LLM at layer 20 (FastV-style,
# last-query attention of layer 19, keep 30%). Positions are contiguous before the LLM, original after pruning.
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Optional, Tuple, Union, List

@dataclass
class FlashVidConfig:
    retention_ratio: float = 0.25
    alpha: float = 0.7
    token_selection_method: str = "attn_div_v2"
    temporal_threshold: float = 0.8
    do_segment: bool = True
    segment_threshold: float = 0.9
    min_segment_num: int = 8
    complementary_segment: bool = True
    num_attn_div_tokens: Optional[int] = None
    num_sttm_tokens: Optional[int] = None
    visual_token_start_index: Optional[int] = None
    visual_token_length: Optional[int] = None
    expansion: float = 1.25
    pruning_layer: int = 20
    llm_retention_ratio: float = 0.3

def pairwise_cosine_distances(image_features):
    normed_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    similarities = torch.bmm(normed_features, normed_features.transpose(-1, -2))
    return 1.0 - similarities

def attn_div_v2_based_token_selection(features, cls_attention, num_retained_tokens):
    original_features = features
    features = features.float()
    pooled_features = features.mean(1)
    global_cls_attention = cls_attention.float() * 1e6
    bsz, num_visual_tokens, feat_dim = features.shape
    dist_matrix = pairwise_cosine_distances(features)
    calibration_term1 = global_cls_attention.unsqueeze(1)
    local_cls_attention = torch.einsum("b n d, c d -> b c n", features, pooled_features).mean(1)
    calibration_term2 = local_cls_attention.unsqueeze(1)
    dist_matrix = dist_matrix * calibration_term1 * calibration_term2
    keep_indices = torch.zeros(bsz, num_retained_tokens, dtype=torch.long, device=features.device)
    min_dist = torch.topk(dist_matrix, k=2, dim=1, largest=False).values[:, 1, :]
    keep_indices[:, 0] = torch.argmax(min_dist, dim=-1)
    for i in range(1, num_retained_tokens):
        dist_sub_matrix = torch.gather(dist_matrix, dim=1,
                                       index=keep_indices[:, :i].unsqueeze(-1).expand(-1, -1, num_visual_tokens))
        min_dist = torch.min(dist_sub_matrix, dim=1).values
        keep_indices[:, i] = torch.argmax(min_dist, dim=-1)
    keep_indices = keep_indices.sort().values
    selected_features = torch.gather(original_features, dim=1,
                                     index=keep_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, keep_indices

def attn_based_token_selection(features, cls_attention, num_retained_tokens):
    bsz, num_visual_tokens, feat_dim = features.shape
    topk_indices = torch.topk(cls_attention, k=num_retained_tokens, dim=-1).indices.sort().values
    selected_features = torch.gather(features, dim=1, index=topk_indices.unsqueeze(-1).expand(-1, -1, feat_dim))
    return selected_features, topk_indices

ALL_TOKEN_SELECTION_METHOD = {"attn_div_v2": attn_div_v2_based_token_selection, "attn": attn_based_token_selection}

def flashvid_compression(video_features, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    if flashvid_config.do_segment:
        segment_lengths = fv_segment(video_features=video_features.mean(1),
                                     segment_threshold=flashvid_config.segment_threshold,
                                     min_segment_num=flashvid_config.min_segment_num,
                                     complementary_segment=flashvid_config.complementary_segment)
    else:
        segment_lengths = torch.tensor([num_frames], dtype=torch.long, device=video_features.device)
    num_segments = segment_lengths.shape[0]
    global_indices = torch.arange(num_frames * num_visual_tokens, dtype=torch.long, device=video_features.device)
    token_budget = math.ceil(num_visual_tokens * flashvid_config.retention_ratio * flashvid_config.expansion)
    num_attn_div_tokens = math.ceil(token_budget * flashvid_config.alpha)
    num_sttm_tokens = token_budget - num_attn_div_tokens
    flashvid_config.num_attn_div_tokens = num_attn_div_tokens
    flashvid_config.num_sttm_tokens = num_sttm_tokens
    all_segment_features, all_segment_indices = [], []
    offset = 0
    for seg_idx in range(num_segments):
        seg_len = segment_lengths[seg_idx]
        segment_features = video_features[offset: offset + seg_len]
        segment_cls_attention = cls_attention[offset: offset + seg_len]
        segment_global_indices = global_indices.view(num_frames, num_visual_tokens)[offset: offset + seg_len]
        segment_features, segment_global_indices = fv_segment_compression(
            segment_features=segment_features, segment_global_indices=segment_global_indices,
            cls_attention=segment_cls_attention, flashvid_config=flashvid_config)
        all_segment_features.append(segment_features)
        all_segment_indices.append(segment_global_indices)
        offset += seg_len
    final_tokens = torch.cat(all_segment_features, dim=0)
    final_global_indices = torch.cat(all_segment_indices, dim=0)
    sorted_indices = final_global_indices.argsort()
    sorted_tokens = final_tokens[sorted_indices]
    flashvid_config.visual_token_length = sorted_tokens.shape[0]
    return sorted_tokens, final_global_indices[sorted_indices]

def fv_segment_compression(segment_features, segment_global_indices, cls_attention, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = segment_features.shape
    if flashvid_config.alpha > 0:
        additional_kwargs = {"cls_attention": cls_attention} if "attn" in flashvid_config.token_selection_method else {}
        selected_features, selected_indices = ALL_TOKEN_SELECTION_METHOD[flashvid_config.token_selection_method](
            features=segment_features, num_retained_tokens=flashvid_config.num_attn_div_tokens, **additional_kwargs)
        selected_global_indices = segment_global_indices.gather(1, index=selected_indices).view(-1)
    else:
        selected_features = torch.tensor([]).to(segment_features)
        selected_indices = torch.tensor([]).to(segment_global_indices)
        selected_global_indices = torch.tensor([]).to(segment_global_indices)
    mask = torch.ones(num_frames, num_visual_tokens, dtype=torch.bool, device=segment_features.device)
    mask.scatter_(1, selected_indices, False)
    num_other_tokens = flashvid_config.num_sttm_tokens * num_frames
    if num_other_tokens > 0 and flashvid_config.temporal_threshold < 1.0:
        if num_frames > 1:
            temp_merged_token_list, temp_merged_indices_list = fv_spatiotemporal_compression(
                video_features=segment_features, temporal_threshold=flashvid_config.temporal_threshold,
                token_mask=mask, flashvid_config=flashvid_config)
            temp_merged_global_indices_list = [segment_global_indices.view(num_frames, -1)[i][temp_merged_indices]
                                               for i, temp_merged_indices in enumerate(temp_merged_indices_list)]
        else:
            temp_merged_token_list = [segment_features[0]]
            temp_merged_global_indices_list = [segment_global_indices[0]]
    else:
        temp_merged_token_list = []
        temp_merged_global_indices_list = []
    all_tokens = [selected_features.view(-1, feat_dim)]
    all_global_indices = [selected_global_indices]
    if num_other_tokens > 0:
        num_current_retained_tokens = sum(len(tokens) for tokens in temp_merged_token_list)
        adapative_contextual_ratio = num_other_tokens / num_current_retained_tokens
        if adapative_contextual_ratio < 1.0:
            num_frames_in_segment = len(temp_merged_token_list)
            max_num_tokens = max(len(tokens) for tokens in temp_merged_token_list)
            batched_tokens = torch.zeros((num_frames_in_segment, max_num_tokens, feat_dim),
                                         dtype=segment_features.dtype, device=segment_features.device)
            valid_token_mask = torch.zeros((num_frames_in_segment, max_num_tokens), dtype=torch.bool,
                                           device=segment_features.device)
            num_clusters_list, k_list = [], []
            for i, temp_merged_tokens in enumerate(temp_merged_token_list):
                num_tokens = len(temp_merged_tokens)
                batched_tokens[i, :num_tokens] = temp_merged_tokens
                valid_token_mask[i, :num_tokens] = True
                num_clusters = math.ceil(num_tokens * adapative_contextual_ratio)
                num_clusters_list.append(num_clusters)
                k_list.append(min(num_clusters, 7))
            cluster_indices_list, cluster_center_indices_list = fv_dpc_knn(
                features=batched_tokens, num_clusters=num_clusters_list, k=k_list, valid_token_mask=valid_token_mask)
            for i, (temp_merged_tokens, temp_merged_global_indices) in enumerate(
                    zip(temp_merged_token_list, temp_merged_global_indices_list)):
                num_clusters = num_clusters_list[i]
                if num_clusters > 0:
                    cluster_indices = cluster_indices_list[i][:len(temp_merged_tokens)]
                    cluster_center_indices = cluster_center_indices_list[i]
                    aggregated_tokens = torch.zeros((num_clusters, feat_dim), dtype=segment_features.dtype,
                                                    device=segment_features.device)
                    aggregated_tokens.scatter_add_(0, cluster_indices.unsqueeze(-1).expand(-1, feat_dim),
                                                   temp_merged_tokens)
                    cluster_counts = torch.bincount(cluster_indices, minlength=num_clusters).unsqueeze(-1).to(
                        segment_features.dtype)
                    aggregated_tokens = aggregated_tokens / cluster_counts
                    global_token_indices = temp_merged_global_indices[cluster_center_indices]
                else:
                    aggregated_tokens = temp_merged_tokens
                    global_token_indices = temp_merged_global_indices
                all_tokens.append(aggregated_tokens)
                all_global_indices.append(global_token_indices)
        else:
            for temp_merged_tokens, temp_merged_global_indices in zip(temp_merged_token_list,
                                                                      temp_merged_global_indices_list):
                all_tokens.append(temp_merged_tokens)
                all_global_indices.append(temp_merged_global_indices)
    segment_final_tokens = torch.cat(all_tokens, dim=0)
    segment_final_global_indices = torch.cat(all_global_indices, dim=0)
    return segment_final_tokens, segment_final_global_indices

def fv_segment(video_features, segment_threshold, min_segment_num, complementary_segment=True):
    num_frames, feat_dim = video_features.shape
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    transition_similarities = torch.sum(normed_video_features[:-1] * normed_video_features[1:], dim=-1)
    cut_indices = torch.where(transition_similarities < segment_threshold)[0]
    return fv_additional_segment(cut_indices=cut_indices, num_frames=num_frames, min_segment_num=min_segment_num,
                                 transition_similarities=transition_similarities,
                                 segment_threshold=segment_threshold, complementary_segment=complementary_segment)

def fv_additional_segment(cut_indices, num_frames, min_segment_num, transition_similarities, segment_threshold,
                          complementary_segment=True):
    num_segments = cut_indices.numel() + 1
    if num_segments < min_segment_num and complementary_segment:
        num_remaining_cut_indices = min_segment_num - num_segments
        transition_similarities[transition_similarities < segment_threshold] = 1.0
        complementary_cut_indices = torch.topk(transition_similarities,
                                               k=min(num_remaining_cut_indices, transition_similarities.shape[0]),
                                               largest=False).indices
        cut_indices = torch.cat([cut_indices, complementary_cut_indices]).sort().values
    padded_cut_indices = F.pad(cut_indices, (1, 1), value=0)
    padded_cut_indices[0] = -1
    padded_cut_indices[-1] = num_frames - 1
    return torch.diff(padded_cut_indices, n=1, dim=0)

@torch.no_grad()
def fv_dpc_knn(features, num_clusters, k=7, valid_token_mask=None):
    invalid_token_mask = ~valid_token_mask if valid_token_mask is not None else None
    bsz, seq_len, feat_dim = features.shape
    dists = torch.cdist(features.float(), features.float()) / math.sqrt(feat_dim)
    if valid_token_mask is not None:
        dists = torch.masked_fill(dists, invalid_token_mask.unsqueeze(1).expand(-1, seq_len, -1), dists.max() + 1)
    max_k = max(k) if isinstance(k, list) else k
    nearest_dist = torch.topk(dists, k=max_k, dim=-1, largest=False).values
    if isinstance(k, list):
        density = torch.empty((bsz, seq_len), device=features.device, dtype=features.dtype)
        for i in range(bsz):
            density[i] = torch.mean(-(nearest_dist[i, :, :k[i]] ** 2), dim=-1).exp()
    else:
        density = torch.mean(-(nearest_dist ** 2), dim=-1).exp()
    if valid_token_mask is not None:
        density = torch.masked_fill(density, invalid_token_mask, 0.0)
    mask = density[:, None, :] > density[:, :, None]
    max_dist = dists.view(bsz, -1).max(dim=-1)[0].view(-1, 1, 1)
    modified_dists = torch.where(mask, dists, max_dist)
    dist, _ = torch.min(modified_dists, dim=-1)
    score = dist * density
    if isinstance(num_clusters, int):
        cluster_center_indices = torch.topk(score, k=num_clusters, dim=-1).indices
        dists = torch.gather(dists, dim=-1, index=cluster_center_indices.unsqueeze(1).expand(-1, seq_len, -1))
        cluster_indices = torch.argmin(dists, dim=-1)
        cluster_indices.scatter_(dim=-1, index=cluster_center_indices,
                                 src=torch.arange(num_clusters).to(cluster_indices).unsqueeze(0).expand(bsz, -1))
        return cluster_indices, cluster_center_indices
    cluster_indices_list, cluster_center_indices_list = [], []
    for i in range(bsz):
        k_i = num_clusters[i]
        if k_i == 0:
            cluster_center_indices_list.append(torch.tensor([], dtype=torch.long, device=features.device))
            cluster_indices_list.append(torch.zeros(seq_len, dtype=torch.long, device=features.device))
            continue
        cc_idx = torch.topk(score[i], k=k_i, dim=-1).indices
        cluster_center_indices_list.append(cc_idx)
        dists_i = torch.gather(dists[i], dim=-1, index=cc_idx.unsqueeze(0).expand(seq_len, -1))
        c_idx = torch.argmin(dists_i, dim=-1)
        c_idx.scatter_(dim=-1, index=cc_idx, src=torch.arange(k_i).to(c_idx))
        cluster_indices_list.append(c_idx)
    return cluster_indices_list, cluster_center_indices_list

def fv_spatiotemporal_compression(video_features, temporal_threshold, token_mask, flashvid_config):
    num_frames, num_visual_tokens, feat_dim = video_features.shape
    lower_bound = (flashvid_config.num_attn_div_tokens + flashvid_config.num_sttm_tokens) * num_frames
    normed_video_features = video_features / video_features.norm(p=2, dim=-1, keepdim=True)
    cosine_similarities = torch.bmm(normed_video_features[1:], normed_video_features[:-1].transpose(1, 2))
    cosine_similarities[~token_mask[1:].unsqueeze(-1).expand(-1, -1, num_visual_tokens)] = -1.0
    cosine_similarities[~token_mask[:-1].unsqueeze(1).expand(-1, num_visual_tokens, -1)] = -1.0
    max_sims, max_sim_indices = torch.max(cosine_similarities, dim=-1)
    padded_max_sims = F.pad(max_sims, (0, 0, 1, 0), value=-1)
    padded_max_sim_indices = F.pad(max_sim_indices, (0, 0, 1, 0), value=-1)
    token_counts = torch.ones(num_frames, num_visual_tokens).to(video_features)
    mask = padded_max_sims > temporal_threshold
    retaining_token_mask = ~mask
    if retaining_token_mask.int().sum() < lower_bound:
        soft_threshold = padded_max_sims.view(-1).topk(k=(num_frames * num_visual_tokens) - lower_bound).values[-1]
        soft_threshold = max(soft_threshold, -1.0 + 1e-6)
        mask = padded_max_sims > soft_threshold
        retaining_token_mask = ~mask
    for frame_idx in range(num_frames - 1, -1, -1):
        frame_features = video_features[frame_idx]
        frame_token_counts = token_counts[frame_idx]
        frame_max_sim_indices = padded_max_sim_indices[frame_idx]
        tokens_to_merge = frame_features[~mask[frame_idx]]
        to_merge_token_counts = frame_token_counts[~mask[frame_idx]]
        if tokens_to_merge.numel() > 0:
            aggregated_tokens = tokens_to_merge / to_merge_token_counts.unsqueeze(-1).to(tokens_to_merge.dtype)
            video_features[frame_idx][~mask[frame_idx]] = aggregated_tokens
            token_counts[frame_idx][~mask[frame_idx]] = 1
        other_tokens = frame_features[mask[frame_idx]]
        if other_tokens.numel() > 0:
            anchor_token_indices = frame_max_sim_indices[mask[frame_idx]]
            aggregated_tokens = torch.zeros((num_visual_tokens, feat_dim), dtype=video_features.dtype,
                                            device=video_features.device)
            aggregated_tokens.scatter_add_(0, anchor_token_indices.unsqueeze(-1).expand(-1, feat_dim), other_tokens)
            aggregated_token_counts = torch.bincount(anchor_token_indices, minlength=num_visual_tokens).to(
                video_features.dtype)
            video_features[frame_idx - 1] += aggregated_tokens
            token_counts[frame_idx - 1] += aggregated_token_counts
            token_counts[frame_idx][mask[frame_idx]] = 0
    final_tokens, retained_token_indices = [], []
    for i in range(num_frames):
        frame_mask = retaining_token_mask[i] & token_mask[i]
        final_tokens.append(video_features[i][frame_mask])
        retained_token_indices.append(torch.where(frame_mask)[0])
    return final_tokens, retained_token_indices

@torch.no_grad()
def flashvid_compress(X, cls_attn, r):
    cfg = FlashVidConfig(retention_ratio=r, **FLASHVID)
    # official code edits the pooled features in place, so work on a copy; the attention enters in the model dtype
    toks, idx = flashvid_compression(X.clone(), cls_attn.to(X.device, X.dtype), cfg)
    inner = {"kind": "fastv", "layer": cfg.pruning_layer, "keep_ratio": cfg.llm_retention_ratio,
             "include_newline": False}
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": True, "inner": inner}


# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# HoliTom (Shao et al., NeurIPS 2025) — github.com/cokeshao/HoliTom, holitom/llava_arch.py + modeling_qwen2.py
# Apache-2.0. Methods copied as plain functions (self removed); settings from scripts/eval_ov-7b_holitom.sh:
# T=0.65 at 10% (0.80 otherwise), BETA=0.6 (beta weighting on, as the script leaves NO_BETA unset), D=0, K=7,
# inner-LLM merging at layer 18 with r=0.5 (HOLITOM_k=18, HOLITOM_r=0.5). Only edit: the 1e-6 density tie-break noise
# uses a seeded generator. LLaVA-OV uses mm_newline_position "one_token", so tokens are [static, dynamic] per window,
# one newline at the end, contiguous positions; the inner merge treats the newline as part of the image span (as there).
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
def ht_cluster_dpc_knn(x, cluster_num, k=7, gen=None):
    with torch.no_grad():
        batch_size, seq_len, embed_dim = x.shape
        dist_matrix = torch.cdist(x.float(), x.float()) / (embed_dim ** 0.5)
        dist_nearest, index_nearest = torch.topk(dist_matrix, k, dim=-1, largest=False)
        density = (-(dist_nearest ** 2).mean(dim=-1)).exp()
        density = density + torch.rand(density.shape, device=density.device, dtype=density.dtype, generator=gen) * 1e-6
        mask = (density[:, None, :] > density[:, :, None]).type(x.dtype)
        dist_max = dist_matrix.flatten(1).max(dim=-1).values[:, None, None]
        dist, index_parent = (dist_matrix * mask + dist_max * (1 - mask)).min(dim=-1)
        score = dist * density
        _, index_center = score.topk(cluster_num, dim=-1)
        return index_center, dist_matrix

def ht_select_static_windows(feature_sim, batch_size, tau, max_window_size):
    def get_pruned_static_count_vectorized(feature_sim, batch_size, tau):
        similarity_matrix = torch.ones((batch_size, batch_size, feature_sim.shape[1]), device=feature_sim.device)
        for start in range(batch_size - 1):
            cum_similarity = torch.cumprod(feature_sim[start:] > tau, dim=0)
            similarity_matrix[start, start + 1:start + 1 + len(cum_similarity)] = cum_similarity
        window_lengths = torch.arange(batch_size, device=feature_sim.device).unsqueeze(0) - \
            torch.arange(batch_size, device=feature_sim.device).unsqueeze(1)
        window_lengths = window_lengths.clamp(min=0)
        return (similarity_matrix.sum(dim=-1) * window_lengths).float()
    pruned_static_count = get_pruned_static_count_vectorized(feature_sim, batch_size, tau)
    dp = torch.zeros(batch_size, device=pruned_static_count.device)
    prev = torch.zeros(batch_size, dtype=torch.long, device=pruned_static_count.device)
    for i in range(batch_size):
        max_val = dp[i - 1] if i > 0 else 0
        best_j = i
        for window_size in range(2, min(i + 1, max_window_size) + 1):
            j = i - window_size
            current_val = (dp[j] if j >= 0 else 0) + pruned_static_count[j + 1, i]
            if current_val > max_val:
                max_val = current_val
                best_j = j + 1
        dp[i] = max_val
        prev[i] = best_j
    selected_frames = []
    i = batch_size - 1
    while i >= 0:
        selected_frames.append((prev[i].item(), i))
        i = prev[i].item() - 1
    selected_frames = selected_frames[::-1]
    return selected_frames, dp[-1].item()

def ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta):
    batch_size, seq_len, embed_dim = feat.shape
    all_indices = torch.arange(seq_len, device=feat.device).unsqueeze(0).expand(batch_size, -1)
    non_target_indices = torch.zeros((batch_size, seq_len - cluster_num), dtype=torch.long, device=feat.device)
    for b in range(batch_size):
        non_target_mask = ~torch.isin(all_indices[b], target_indices[b])
        non_target_indices[b] = all_indices[b][non_target_mask]
    non_target_feat = torch.gather(feat, dim=1, index=non_target_indices.unsqueeze(-1).expand(-1, -1, feat.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=1,
                               index=non_target_indices.unsqueeze(-1).expand(-1, -1, dist_matrix.size(-1)))
    dist_matrix = torch.gather(dist_matrix, dim=2, index=target_indices.unsqueeze(1).expand(-1, dist_matrix.size(1), -1))
    idx_cluster = torch.argmin(dist_matrix, dim=-1)
    cluster_tokens = []
    for b in range(batch_size):
        batch_tokens = []
        for i in range(cluster_num):
            mask = (idx_cluster[b] == i)
            if mask.any():
                cluster_means = non_target_feat[b][mask].mean(dim=0)          # NO_BETA unset -> beta weighting
                batch_tokens.append(Beta * feat[b][target_indices[b][i]] + (1 - Beta) * cluster_means)
            else:
                batch_tokens.append(feat[b][target_indices[b][i]])
        cluster_tokens.append(torch.stack(batch_tokens))
    return torch.stack(cluster_tokens)

def ht_merge_tokens_by_attention_density(feat, attn, pos, retain_ratio, D, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    dominant_num = round(math.ceil(seq_len * retain_ratio) * (1 - D))
    contextual_num = math.ceil(seq_len * retain_ratio) - dominant_num
    if dominant_num > 0:
        all_indices = attn.topk(dominant_num, dim=1).indices
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device).scatter_(1, all_indices, False)
        dominant_tokens = feat.masked_select(~mask.unsqueeze(-1)).view(batch_size, dominant_num, embed_dim)
        dominant_pos = pos.masked_select(~mask).view(batch_size, dominant_num)
    else:
        mask = torch.ones_like(feat[:, :, 0], dtype=torch.bool, device=feat.device)
        dominant_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        dominant_pos = torch.empty((batch_size, 0), device=feat.device)
    if contextual_num > 0:
        feat_filtered = feat.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num, embed_dim)
        contextual_pos = pos.masked_select(mask.unsqueeze(-1)).view(batch_size, seq_len - dominant_num)
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat_filtered, contextual_num, k=min(K, contextual_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        contextual_pos = torch.stack([contextual_pos[b][target_indices[b]] for b in range(batch_size)])
        contextual_tokens = ht_merge_tokens_by_clustering(feat_filtered, target_indices, dist_matrix, contextual_num, Beta)
    else:
        contextual_tokens = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        contextual_pos = torch.empty((batch_size, 0), device=feat.device)
    image_feat, image_pos = [], []
    for b in range(batch_size):
        image_feat.append(torch.cat([dominant_tokens[b], contextual_tokens[b]], dim=0))
        image_pos.append(torch.cat([dominant_pos[b], contextual_pos[b]], dim=0))
    return torch.stack(image_feat), torch.stack(image_pos)

def ht_merge_tokens_by_density(feat, pos, retain_ratio, Beta, K, gen=None):
    batch_size, seq_len, embed_dim = feat.shape
    cluster_num = round(seq_len * retain_ratio)
    if cluster_num > 0:
        target_indices, dist_matrix = ht_cluster_dpc_knn(feat, cluster_num, k=min(K, cluster_num), gen=gen)
        target_indices = torch.sort(target_indices, dim=-1)[0]
        image_pos = torch.stack([pos[b][target_indices[b]] for b in range(batch_size)])
        image_feat = ht_merge_tokens_by_clustering(feat, target_indices, dist_matrix, cluster_num, Beta)
    else:
        image_feat = torch.empty((batch_size, 0, embed_dim), device=feat.device)
        image_pos = torch.empty((batch_size, 0), device=feat.device)
    return image_feat, image_pos

def ht_holitom(static_feat, dynamic_feat, dynamic_attn, static_pos, dynamic_pos, window_size, retain_ratio, D, Beta,
               K, images_dtype, gen=None):
    # mm_newline_position == "one_token" branch (LLaVA-OneVision)
    if window_size == 1:
        dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                         retain_ratio, D, Beta, K, gen)
        return dynamic_feat.flatten(0, 1).to(images_dtype)
    dynamic_feat, dynamic_pos = ht_merge_tokens_by_attention_density(dynamic_feat, dynamic_attn, dynamic_pos,
                                                                     retain_ratio, D, Beta, K, gen)
    static_feat, static_pos = ht_merge_tokens_by_density(static_feat.unsqueeze(0), static_pos, retain_ratio, Beta, K, gen)
    return torch.cat([static_feat.flatten(0, 1), dynamic_feat.flatten(0, 1)]).to(images_dtype)

def ht_get_static_dynamic_features(image_feat, attn_weights, selected_frames, feature_sim, tau):
    batch_size, seq_len, embed_dim = image_feat.shape
    static_feat_list, dynamic_feat_list, dynamic_attn_list, static_pos_list, dynamic_pos_list = [], [], [], [], []
    for start, end in selected_frames:
        all_indices = torch.arange(seq_len, device=image_feat.device).unsqueeze(0)
        if start == end:
            static_feat_list.append(torch.empty((0, embed_dim), device=image_feat.device))
            dynamic_feat_list.append(image_feat[start:end + 1])
            dynamic_attn_list.append(attn_weights[start:end + 1])
            static_pos_list.append(torch.empty((0, seq_len), device=image_feat.device))
            dynamic_pos_list.append(all_indices)
        else:
            windows_size = end - start + 1
            mask = torch.all(feature_sim[start:end, :] > tau, dim=0)
            static_feat_list.append(image_feat[start:end + 1, mask].mean(dim=0))
            dynamic_feat_list.append(image_feat[start:end + 1, ~mask])
            dynamic_attn_list.append(attn_weights[start:end + 1, ~mask])
            static_pos_list.append(all_indices[:, mask].expand(1, -1))
            dynamic_pos_list.append(all_indices[:, ~mask].expand(windows_size, -1))
    return static_feat_list, dynamic_feat_list, None, dynamic_attn_list, static_pos_list, dynamic_pos_list

@torch.no_grad()
def holitom_compress(X, attn, r, seed):
    """X: (T, S, D) pooled projected tokens; attn: (T, S) pooled last-vision-layer attention (mean heads, queries)."""
    tau = HOLITOM["T"][r]
    Beta, D, K, max_window_size = HOLITOM["beta"], HOLITOM["D"], HOLITOM["K"], HOLITOM["max_window"]
    gen = torch.Generator(device=X.device).manual_seed(seed)
    pooled_image_feat = X
    attn_weights = attn.to(X.device, X.dtype)
    batch_size, seq_len, embed_dim = pooled_image_feat.shape
    normed = torch.nn.functional.normalize(pooled_image_feat, p=2, dim=-1)
    feature_sim = torch.nn.functional.cosine_similarity(normed[:-1], normed[1:], dim=-1)
    selected_frames, total_reduced = ht_select_static_windows(feature_sim, batch_size, tau, max_window_size)
    total_tokens = batch_size * seq_len
    retain_ratio = min(r / ((total_tokens - total_reduced) / total_tokens), 1)
    sf, df, _, da, sp, dpos = ht_get_static_dynamic_features(pooled_image_feat, attn_weights, selected_frames,
                                                             feature_sim, tau)
    segs = []
    for i, (start, end) in enumerate(selected_frames):
        segs.append(ht_holitom(sf[i], df[i], da[i], sp[i], dpos[i], end - start + 1, retain_ratio, D, Beta, K,
                               X.dtype, gen))
    toks = torch.cat(segs, dim=0)
    inner = {"kind": "holitom", "layer": HOLITOM["inner_layer"], "merge_ratio": HOLITOM["inner_r"],
             "include_newline": True}
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": True, "inner": inner}


# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# LLaVA-Scissor (Sun et al., 2025) — github.com/HumanMLLM/LLaVA-Scissor, llava/model/llava_arch_zip.py
# (UnionFind + approximate_components copied verbatim; spatial SCC per frame, temporal SCC, then every original
# token merged into its most similar component token). As in that code, the video path is forced to "flat":
# no newline token, tokens in component order, contiguous positions. Its paper sets the budget only through the
# similarity threshold tau; to compare methods at the SAME budget per video we search tau per video so that the
# output size is as close as possible to round(r * T * S) (documented deviation). Merging sums are done in fp32.
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
from collections import defaultdict

class UnionFind:
    def __init__(self, size):
        self.parent = np.arange(size, dtype=np.int64)
        self.rank = np.zeros(size, dtype=np.int32)
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def batch_union(self, x_arr, y_arr):
        for x, y in zip(x_arr, y_arr):
            x_root = self.find(x)
            y_root = self.find(y)
            if x_root == y_root:
                continue
            if self.rank[x_root] < self.rank[y_root]:
                self.parent[x_root] = y_root
            else:
                self.parent[y_root] = x_root
                if self.rank[x_root] == self.rank[y_root]:
                    self.rank[x_root] += 1

def approximate_components(adj_matrix, epsilon=0.05, rng=np.random):
    n = adj_matrix.shape[0]
    all_nodes = np.ones(n)
    all_indices = np.arange(0, n)
    if n == 0:
        return []
    sample_size = min(n, int(np.ceil(np.log(n) / epsilon ** 2)))
    sampled_nodes = rng.choice(n, size=sample_size, replace=False)
    all_nodes[sampled_nodes] = 0
    neighbor_dict = defaultdict(list)
    for i in sampled_nodes:
        neighbors = np.nonzero(adj_matrix[i])[0]
        valid_neighbors = np.intersect1d(neighbors, all_indices, assume_unique=True)
        neighbor_dict[i] = valid_neighbors
        all_nodes[neighbors] = 0
    remain_nodes = np.nonzero(all_nodes)[0]
    remain_nodes = [[element] for element in remain_nodes]
    uf = UnionFind(n)
    all_x, all_y = [], []
    for i in sampled_nodes:
        for j in neighbor_dict[i]:
            all_x.append(i)
            all_y.append(j)
    uf.batch_union(np.array(all_x), np.array(all_y))
    sampled_roots = np.array([uf.find(i) for i in sampled_nodes])
    unique_roots, counts = np.unique(sampled_roots, return_counts=True)
    components = []
    for root in unique_roots:
        cluster = np.where(uf.parent == root)[0].tolist()
        if len(cluster) > 0:
            components.append(cluster)
    components.extend(remain_nodes)
    degrees = np.count_nonzero(adj_matrix, axis=1)
    def get_sort_key(cluster):
        max_degree = -1
        min_node = float("inf")
        for node in cluster:
            current_degree = degrees[node]
            if (current_degree > max_degree) or (current_degree == max_degree and node < min_node):
                max_degree = current_degree
                min_node = node
        return min_node
    components.sort(key=get_sort_key)
    return components

def _scissor_once(Xf, tau, eps, rng):
    """Xf: (T, S, D) float32. Returns (selected tokens float32, count). Same steps as the official code."""
    T, S, D = Xf.shape
    nrm = Xf / torch.norm(Xf, p=2, dim=-1, keepdim=True)
    high = (torch.matmul(nrm, nrm.transpose(1, 2)) > tau).cpu().numpy()
    fused = []
    for f in range(T):
        for comp in approximate_components(high[f], epsilon=eps, rng=rng):
            fused.append(Xf[f, comp].mean(dim=0))
    sel = torch.stack(fused)
    sn = sel / torch.norm(sel, p=2, dim=1, keepdim=True)
    high2 = (sn @ sn.t() > tau).cpu().numpy()
    sel = torch.stack([sel[comp].mean(dim=0) for comp in approximate_components(high2, epsilon=eps, rng=rng)])
    return sel

def _scc_count_fast(Xf, tau):
    """Exact connected-component count with scipy (used only to choose tau quickly)."""
    from scipy.sparse.csgraph import connected_components
    from scipy.sparse import csr_matrix
    T, S, D = Xf.shape
    nrm = Xf / torch.norm(Xf, p=2, dim=-1, keepdim=True)
    high = (torch.matmul(nrm, nrm.transpose(1, 2)) > tau).cpu().numpy()
    means = []
    for f in range(T):
        k, lab = connected_components(csr_matrix(high[f]), directed=False)
        lab_t = torch.as_tensor(lab, device=Xf.device)
        m = torch.zeros(k, D, device=Xf.device).index_add_(0, lab_t, Xf[f])
        means.append(m / torch.bincount(lab_t, minlength=k).unsqueeze(1).float())
    sel = torch.cat(means)
    sn = sel / torch.norm(sel, p=2, dim=1, keepdim=True)
    k2, _ = connected_components(csr_matrix((sn @ sn.t() > tau).cpu().numpy()), directed=False)
    return k2

@torch.no_grad()
def scissor_compress(X, r, seed):
    T, S, D = X.shape
    Xf = X.float()
    target = max(1, round(r * T * S))
    eps = SCISSOR["epsilon"]
    # 1) quick bracket with an exact connected-component count (scipy)
    lo, hi = -1.0, 1.0
    for _ in range(SCISSOR["search_steps"]):
        mid = (lo + hi) / 2
        if _scc_count_fast(Xf, mid) < target:
            lo = mid
        else:
            hi = mid
    tau0 = hi
    # 2) refine with the official (sampled, approximate) algorithm itself, same seed at every tau
    cache = {}
    def run(t):
        t = round(t, 6)
        if t not in cache:
            cache[t] = _scissor_once(Xf, t, eps, np.random.RandomState(seed))
        return cache[t]
    lo, hi = max(-1.0, tau0 - 0.1), min(1.0, tau0 + 0.02)
    while run(lo).shape[0] > target and lo > -1.0:
        lo = max(-1.0, lo - 0.1)
    while run(hi).shape[0] < target and hi < 1.0:
        hi = min(1.0, hi + 0.05)
    for _ in range(SCISSOR["refine_steps"]):
        mid = (lo + hi) / 2
        if run(mid).shape[0] < target:
            lo = mid
        else:
            hi = mid
    best = min([lo, hi], key=lambda t: abs(run(t).shape[0] - target))
    sel = run(best)
    # merge every original token into its most similar selected token (official final step)
    rem = Xf.reshape(-1, D)
    simm = (rem / torch.norm(rem, p=2, dim=1, keepdim=True)) @ (sel / torch.norm(sel, p=2, dim=1, keepdim=True)).t()
    closest = torch.argmax(simm, dim=1)
    merged = torch.zeros_like(sel).scatter_add_(0, closest.view(-1, 1).expand(-1, D), rem)
    counts = torch.bincount(closest, minlength=sel.shape[0]).float() + 1
    toks = ((sel + merged) / counts.unsqueeze(1)).to(X.dtype)
    return {"tokens": toks, "pos": np.arange(toks.shape[0]), "newline": False, "inner": None, "tau": best}


def compress_all(X, VA, seed):
    """Every (method, setting) -> dict(tokens, pos, n_full, newline, inner). Official code is kept as is, so if it
    fails on a video (e.g. FlashVID's fp16 density can underflow on unusual inputs) that (method, setting) is skipped
    for this video and reported — never silently replaced."""
    T, L, D = X.shape
    out, failures = {}, []
    jobs = ([("flashvid", r, lambda r=r: flashvid_compress(X, VA, r)) for r in METHOD_GRID["flashvid"]] +
            [("holitom", r, lambda r=r: holitom_compress(X, VA, r, seed)) for r in METHOD_GRID["holitom"]] +
            [("scissor", r, lambda r=r: scissor_compress(X, r, seed)) for r in METHOD_GRID["scissor"]])
    for m, r, fn in jobs:
        try:
            c = fn()
            c["n_full"] = int(c["tokens"].shape[0])
            out[(m, str(r))] = c
        except Exception as e:
            failures.append((m, str(r), f"{type(e).__name__}: {e}"))
    return out, failures

# ── self-test on smooth random "video" features (shapes, finiteness, budgets) ──
def _fake_video(T, S, D, dev, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    base = torch.randn(1, S, D, generator=g)
    drift = torch.cumsum(0.25 * torch.randn(T, 1, D, generator=g), 0)
    return (base + drift + 0.6 * torch.randn(T, S, D, generator=g)).to(dev, torch.float16)

_T, _L = NUM_FRAMES, POOLED * POOLED
_X = _fake_video(_T, _L, 64, EMB_DEVICE)
_VA = torch.rand(_T, _L, device=EMB_DEVICE)
_VA = _VA / _VA.sum(1, keepdim=True)
_comp, _fails = compress_all(_X, _VA, seed=0)
assert not _fails, _fails
for (m, s), c in _comp.items():
    t = c["tokens"]
    assert torch.isfinite(t.float()).all(), (m, s)
    assert len(c["pos"]) == t.shape[0]
    print(f"  {m:9s} {s:5s} enters LLM with {t.shape[0]:5d} tokens = {t.shape[0] / (_T * _L):.3f}"
          + (f" | inner {c['inner']['kind']} @ layer {c['inner']['layer']}" if c["inner"] else "")
          + (f" | tau {c['tau']:.3f}" if "tau" in c else ""))
del _X, _VA, _comp
print("Compressor self-test OK ✅")

### 6 · Main run

In [ ]:
# ── Main loop: per video encode once (+ shifted frames once), compress once; per question run every pass ──
import zlib, re
RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
ERR_PATH = os.path.join(OUT_DIR, "errors.log")
done = set()
if os.path.exists(RES_PATH):
    with open(RES_PATH) as f:
        for line in f:
            try:
                done.add(json.loads(line)["qid"])
            except Exception:
                pass
print(f"{len(done)} questions already in results (resume)")

_PREFIX = re.compile(r"^\s*\(?[A-Ha-h][\.\):]\s*")

def reversed_options(options):
    """Reverse order: new position j shows original option n-1-j, relabelled A, B, C, ..."""
    body = [_PREFIX.sub("", str(o)) for o in options]
    n = len(body)
    return [f"{LETTERS[j]}. {body[n - 1 - j]}" for j in range(n)]

def map_back_reverse(res, n):
    """Express a reversed-order result in original option letters: original i sits at new position n-1-i."""
    lp_new = np.array(res["logp"])
    lp = np.array([lp_new[n - 1 - i] for i in range(n)])
    srt = np.sort(lp)[::-1]
    out = dict(res)
    out.update(pred=LETTERS[int(lp.argmax())], logp=lp.round(4).tolist(), margin=float(srt[0] - srt[1]))
    return out

def run_question(Xf, Xq, comp, row):
    n_opt = min(len(row.options), len(LETTERS))
    suf = build_suffix_ids(row.question, row.options)
    T, L, D = Xf.shape
    def finish(res):
        res["correct"] = bool(res["pred"] == row.answer)
        return res
    rec = {"qid": row.qid, "video": row.video_key, "duration": row.duration, "task_type": row.task_type,
           "gt": row.answer, "n_full": T * L}
    rec["full"] = finish(llm_pass(Xf.reshape(-1, D), np.arange(T * L), T * L, suf, n_opt))
    rec["noise"] = {
        "frame_shift_q": finish(llm_pass(Xq.reshape(-1, D), np.arange(T * L), T * L, suf, n_opt)),
        "option_reverse": finish(map_back_reverse(
            llm_pass(Xf.reshape(-1, D), np.arange(T * L), T * L,
                     build_suffix_ids(row.question, reversed_options(row.options)), n_opt), n_opt)),
    }
    rec["methods"] = {}
    for (m, s), c in comp.items():
        res = finish(llm_pass(c["tokens"], c["pos"], c["n_full"], suf, n_opt, newline=c["newline"], inner=c["inner"]))
        res["retention"] = round(c["tokens"].shape[0] / (T * L), 4)                 # tokens entering the LLM
        res["retention_layer_avg"] = round(res["vis_mean_layers"] / (T * L), 4)     # averaged over LLM layers
        if "tau" in c:
            res["tau"] = round(float(c["tau"]), 5)
        rec["methods"].setdefault(m, {})[s] = res
    return rec

table = load_videomme_table()
T_START = time.time()
time_left = lambda: MAX_RUNTIME_HOURS * 3600 - (time.time() - T_START)
n_new, n_err, consec_fail, CHECK_DONE, stop = 0, 0, 0, False, False

def _fail_guard():
    if consec_fail >= 6:
        raise RuntimeError("6 questions failed in a row — stopping instead of wasting GPU hours. Send me the log.")

for ch in VIDEOMME_CHUNKS:
    if stop or time_left() < 1800:
        print("⏱️ Time budget reached — skipping remaining chunks"); break
    vids = fetch_chunk(ch)
    sub = table[table.video_key.isin(vids.keys()) | table.video_id.isin(vids.keys())]
    sub = sub[~sub.qid.isin(done)]
    if MAX_Q_PER_CHUNK:
        sub = sub.sample(n=min(MAX_Q_PER_CHUNK, len(sub)), random_state=SEED)
    print(f"Chunk {ch}: {len(sub)} questions to run | durations: {sub.duration.value_counts().to_dict()}")
    for vkey, group in sub.groupby("video_key", sort=False):
        if time_left() < 900:
            stop = True; print("⏱️ Time budget reached — stopping"); break
        path = vids.get(vkey) or vids.get(group.video_id.iloc[0])
        t0 = time.time()
        try:
            frames = read_frames(path, NUM_FRAMES)
            if not CHECK_DONE:
                CHECK_DONE = True
                equivalence_check(frames, group.iloc[0])
            Xf, VA = encode_video(preprocess(frames))
            Xq = encode_video(preprocess(read_frames(path, NUM_FRAMES, shift=NOISE_FRAME_SHIFT_Q)))[0]
            tc = time.time()
            comp, fails = compress_all(Xf, VA, seed=zlib.crc32(str(vkey).encode()))
            t_comp = time.time() - tc
            for m, s_, err in fails:
                with open(ERR_PATH, "a") as f: f.write(f"{vkey}\tcompress {m}@{s_}\t{err}\n")
                print(f"  ! {m}@{s_} failed on video {vkey} (skipped for this video): {err}")
        except RuntimeError as e:
            if "Equivalence check" in str(e):
                raise
            n_err += len(group); consec_fail += len(group)
            with open(ERR_PATH, "a") as f: f.write(f"{vkey}\tencode\t{type(e).__name__}: {e}\n")
            print(f"  ✗ video {vkey}: {type(e).__name__}: {e}"); _fail_guard(); continue
        except Exception as e:
            n_err += len(group); consec_fail += len(group)
            with open(ERR_PATH, "a") as f: f.write(f"{vkey}\tencode\t{type(e).__name__}: {e}\n")
            print(f"  ✗ video {vkey}: {type(e).__name__}: {e}"); _fail_guard(); continue
        for row in group.itertuples():
            try:
                rec = run_question(Xf, Xq, comp, row)
                rec["compress_sec_video"] = round(t_comp, 2)
            except torch.cuda.OutOfMemoryError as e:
                torch.cuda.empty_cache(); n_err += 1; consec_fail += 1
                with open(ERR_PATH, "a") as f: f.write(f"{row.qid}\tOOM\t{e}\n")
                print(f"  ✗ {row.qid}: out of GPU memory (skipped)"); _fail_guard(); continue
            except Exception as e:
                n_err += 1; consec_fail += 1
                with open(ERR_PATH, "a") as f: f.write(f"{row.qid}\t{type(e).__name__}\t{e}\n")
                print(f"  ✗ {row.qid}: {type(e).__name__}: {e}"); _fail_guard(); continue
            with open(RES_PATH, "a") as f:
                f.write(json.dumps(rec) + "\n")
            done.add(row.qid); n_new += 1; consec_fail = 0
            full = rec["full"]["pred"]
            flips = sum(r["pred"] != full for m in rec["methods"].values() for r in m.values())
            nz = rec["noise"]
            print(f"  ✓ {row.qid} [{row.duration}] gt={row.answer} full={full} "
                  f"shiftQ={nz['frame_shift_q']['pred']} rev={nz['option_reverse']['pred']} "
                  f"| {flips}/{sum(len(v) for v in rec['methods'].values())} compressed passes changed "
                  f"| done {len(done)} | {time.time()-t0:.0f}s | {time_left()/3600:.1f} h left")
        del Xf, Xq, VA, comp
        gc.collect(); torch.cuda.empty_cache()
    cleanup_chunk(ch)
print(f"\nFinished: {n_new} new questions, {n_err} errors, {len(done)} total in {RES_PATH}")

### 7 · Gate verdicts and noise passes

In [ ]:
# ── Step 1b analysis: gate verdicts, held-out noise passes, flips ──
import glob
import matplotlib.pyplot as plt

RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
R = []
with open(RES_PATH) as f:
    for line in f:
        try:
            R.append(json.loads(line))
        except Exception:
            pass
# Optional: Step 1 results attached as a Kaggle input -> add its two noise passes and methods for the same questions
STEP1 = {}
for p in glob.glob("/kaggle/input/**/results.jsonl", recursive=True):
    try:
        with open(p) as f:
            for line in f:
                r1 = json.loads(line)
                if "option_cycle" in r1.get("noise", {}):
                    STEP1[r1["qid"]] = r1
    except Exception as e:
        print("Could not read", p, e)
if STEP1:
    same = [r for r in R if r["qid"] in STEP1]
    agree = np.mean([STEP1[r["qid"]]["full"]["pred"] == r["full"]["pred"] for r in same]) if same else float("nan")
    print(f"Step 1 results found: {len(STEP1)} questions; {len(same)} overlap; full-model answers identical "
          f"across the two runs on {100 * agree:.1f}% (run-to-run determinism check)")
    for r in R:
        r1 = STEP1.get(r["qid"])
        if r1:
            r["noise"].update(r1["noise"])
            for m, v in r1["methods"].items():
                r["methods"].setdefault(m, v)
    R = [r for r in R if len(r["noise"]) == 4] if len(same) >= 0.9 * len(R) else R
else:
    print("Step 1 results not attached — analysing Step 1b passes only (that is fine).")
n = len(R)
assert n >= 10, f"Only {n} results — nothing to analyse yet."
rng = np.random.default_rng(SEED)
vid = np.array([r["video"] for r in R])
videos = np.unique(vid)
v_rows = [np.flatnonzero(vid == v) for v in videos]
full_pred = np.array([r["full"]["pred"] for r in R])
full_cor = np.array([r["full"]["correct"] for r in R])
full_lp = [np.array(r["full"]["logp"]) for r in R]
B = 2000
boot_idx = [np.concatenate([v_rows[j] for j in rng.integers(0, len(videos), len(videos))]) for _ in range(B)]

def kl(p_lp, q_lp):
    p = np.exp(p_lp)
    return float((p * (p_lp - q_lp)).sum())

def stats(get):
    """Metrics on the questions where this pass exists (a method can be skipped on a video it failed on)."""
    ok = np.array([get(r) is not None for r in R])
    idx = np.flatnonzero(ok)
    pred = np.array([get(R[k])["pred"] for k in idx]); cor = np.array([get(R[k])["correct"] for k in idx])
    fp, fc = full_pred[idx], full_cor[idx]
    pos_of = -np.ones(n, dtype=int); pos_of[idx] = np.arange(len(idx))
    bsub = [pos_of[i][ok[i]] for i in boot_idx]
    rel = 100 * cor.mean() / fc.mean()
    rel_b = np.array([100 * cor[i].mean() / max(fc[i].mean(), 1e-9) for i in bsub])
    flip = pred != fp
    flip_b = np.array([flip[i].mean() for i in bsub])
    return {
        "coverage": float(ok.mean()),
        "acc": float(cor.mean()), "rel_acc": float(rel),
        "rel_ci": [float(np.percentile(rel_b, 2.5)), float(np.percentile(rel_b, 97.5))],
        "flip": float(flip.mean()), "flip_ci": [float(np.percentile(flip_b, 2.5)), float(np.percentile(flip_b, 97.5))],
        "harmful": float(((~cor) & fc).mean()), "helpful": float((cor & ~fc).mean()),
        "kl": float(np.mean([kl(full_lp[k], np.array(get(R[k])["logp"])) for k in idx])),
        "flops_ratio": float(np.mean([get(R[k])["flops"] for k in idx]) / np.mean([R[k]["full"]["flops"] for k in idx])),
        "time_ratio": float(np.mean([get(R[k])["sec"] for k in idx]) / np.mean([R[k]["full"]["sec"] for k in idx])),
        "peak_gb": float(np.max([get(R[k]).get("peak_gb", np.nan) for k in idx])),
    }

print(f"=== {n} questions, {len(videos)} videos | full-token accuracy {full_cor.mean():.3f} "
      f"| full peak memory {np.max([r['full'].get('peak_gb', np.nan) for r in R]):.1f} GB ===\n")

# ---------------- noise floor ----------------
NOISE_KEYS = [k for k in R[0]["noise"] if all(k in r["noise"] for r in R)]
noise = {k: stats(lambda r, k=k: r["noise"][k]) for k in NOISE_KEYS}
print("Noise floor (full model vs itself):")
for k, s in noise.items():
    print(f"   {k:13s} flips {s['flip']:.3f} (95% CI {s['flip_ci'][0]:.3f}–{s['flip_ci'][1]:.3f}) "
          f"| rel acc {s['rel_acc']:.1f}% | harmful/helpful {s['harmful']:.3f}/{s['helpful']:.3f} | KL {s['kl']:.3f}")

# ---------------- methods + gate ----------------
rows = []
METHODS = {}
for r in R:
    for m, v in r["methods"].items():
        for s in v:
            if s not in METHODS.setdefault(m, []):
                METHODS[m].append(s)
for m, settings in METHODS.items():
    for s in settings:
        get = lambda r, m=m, s=s: r["methods"].get(m, {}).get(s)
        st = stats(get)
        have = [get(r) for r in R if get(r) is not None]
        st["retention"] = float(np.mean([x["retention"] for x in have]))
        st["retention_layer_avg"] = float(np.mean([x.get("retention_layer_avg", x["retention"]) for x in have]))
        ref = GATE_REFS.get((m, s))
        if ref is None:
            verdict, ref_txt = ("no reference" if m in METHOD_GRID else "Step 1"), ""
        else:
            lo, hi = min(ref["values"]), max(ref["values"])
            near = (lo - GATE_TOL_POINTS) <= st["rel_acc"] <= (hi + GATE_TOL_POINTS)
            covers = st["rel_ci"][0] <= hi and st["rel_ci"][1] >= lo
            verdict = "PASS" if (near or covers) else "FAIL"
            ref_txt = "/".join(f"{v:.1f}" for v in ref["values"])
        hh = st["harmful"] + st["helpful"]
        rows.append({"method": m, "setting": s, "coverage": round(st["coverage"], 3), "retention": round(st["retention"], 4),
                     "retention_layer_avg": round(st["retention_layer_avg"], 4),
                     "harmful_share": round(st["harmful"] / hh, 3) if hh > 0 else float("nan"),
                     "acc": round(st["acc"], 4), "rel_acc_%": round(st["rel_acc"], 1),
                     "rel_ci_%": f"{st['rel_ci'][0]:.1f}–{st['rel_ci'][1]:.1f}", "published_%": ref_txt,
                     "gate": verdict, "flip": round(st["flip"], 4),
                     "flip_ci": f"{st['flip_ci'][0]:.3f}–{st['flip_ci'][1]:.3f}",
                     "harmful": round(st["harmful"], 4), "helpful": round(st["helpful"], 4),
                     "kl": round(st["kl"], 4), "flops_ratio": round(st["flops_ratio"], 4),
                     "time_ratio": round(st["time_ratio"], 4), "peak_gb": round(st["peak_gb"], 2)})
gate = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\nMethods (relative accuracy = method / full on the same questions):")
print(gate[["method", "setting", "coverage", "retention", "retention_layer_avg", "rel_acc_%", "rel_ci_%", "published_%", "gate",
            "flip", "harmful_share", "kl", "flops_ratio", "time_ratio"]].to_string(index=False))
gate.to_csv(os.path.join(OUT_DIR, "gate_table.csv"), index=False)

method_verdict = {}
for m, g in gate.groupby("method"):
    v = set(g.gate) - {"no reference", "Step 1"}
    method_verdict[m] = ("from Step 1" if "Step 1" in set(g.gate) else "no reference") if not v else \
        ("PASS" if v == {"PASS"} else "FAIL")
print("\nGate per method:", method_verdict)

# ---------------- noise passes: overlap and direction (pilot data) ----------------
print("\nNoise passes: share of right<->wrong changes that are right->wrong (harmful share)")
for k, st in noise.items():
    hh = st["harmful"] + st["helpful"]
    print(f"   {k:15s} flips {st['flip']:.3f} | harmful share {st['harmful'] / hh if hh else float('nan'):.2f} | KL {st['kl']:.3f}")
if len(NOISE_KEYS) >= 2:
    fl = {k: np.array([r["noise"][k]["pred"] != r["full"]["pred"] for r in R]) for k in NOISE_KEYS}
    either = np.any(np.stack(list(fl.values())), 0)
    print(f"   questions changed by at least one noise pass: {either.mean():.3f}")

# ---------------- plot ----------------
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
colors = {"fastvid": "tab:blue", "visionzip": "tab:orange", "dycoke": "tab:green", "uniform": "tab:gray",
          "random": "tab:red", "flashvid": "tab:cyan", "holitom": "tab:purple", "scissor": "tab:brown"}
for m, g in gate.groupby("method"):
    ax[0].plot(g.retention, g["rel_acc_%"], "o-", color=colors.get(m), label=m)
    ax[1].plot(g.retention, g.flip, "o-", color=colors.get(m), label=m)
for (m, s), ref in GATE_REFS.items():
    r = gate[(gate.method == m) & (gate.setting == s)]
    if len(r):
        for v in ref["values"]:
            ax[0].plot(r.retention.iloc[0], v, "x", color=colors.get(m), ms=9)
ax[0].axhline(100, c="k", ls="--", lw=0.8); ax[0].set_xscale("log")
ax[0].set_xlabel("token retention"); ax[0].set_ylabel("relative accuracy (%)")
ax[0].set_title("Relative accuracy (o = ours, x = published)"); ax[0].legend(fontsize=8)
for k, ls in zip(NOISE_KEYS, [":", "--", "-.", (0, (1, 3))]):
    ax[1].axhline(noise[k]["flip"], c="k", ls=ls, lw=1, label=f"noise: {k}")
ax[1].set_xscale("log"); ax[1].set_xlabel("token retention"); ax[1].set_ylabel("answers changed vs full")
ax[1].set_title("Answer flips"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "step1b.png"), dpi=150); plt.show()

summary = {"n": n, "n_videos": int(len(videos)), "model": MODEL_ID, "frames": NUM_FRAMES,
           "full_accuracy": float(full_cor.mean()), "noise_floor": noise, "step1_merged": bool(STEP1),
           "gate_per_method": method_verdict, "table": rows}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
if QUICK_TEST:
    print("\n(Quick test: numbers are meaningless — this only proves the pipeline works.)")
print("Files saved in", OUT_DIR)

## What to send back
`summary.json`, `gate_table.csv`, `step1b.png`, `gate_rules.json`, `equivalence_check.json`, `results.jsonl` (and `errors.log` if not empty).
Methods that fail the gate are excluded from the main study and reported as such.